# Lab Work - 2.5


## Q1: Outlier Detection using the Percentile Method

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8')
np.random.seed(42)

# Task 1: Generate 1000 income values from log-normal distribution
income = np.random.lognormal(mean=10, sigma=1.5, size=1000)

# Introduce extreme outliers (very high incomes)
income = np.append(income, [500000, 750000, 1200000, 2500000])

df = pd.DataFrame({'Income': income})
print(f"Original dataset size: {len(df)}")
print(df.describe())

In [ ]:
# Compute 1st and 99th percentiles
p1 = np.percentile(df['Income'], 1)
p99 = np.percentile(df['Income'], 99)

print(f"1st Percentile: {p1:,.2f}")
print(f"99th Percentile: {p99:,.2f}")

In [ ]:
# Remove outliers
df_clean = df[(df['Income'] >= p1) & (df['Income'] <= p99)].copy()

print(f"Dataset size after outlier removal: {len(df_clean)}")
print(f"Number of outliers removed: {len(df) - len(df_clean)}")

In [ ]:
# Plot histograms before and after
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.histplot(df['Income'], kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('Income Distribution (With Outliers)')
axes[0].set_xlabel('Income')

sns.histplot(df_clean['Income'], kde=True, ax=axes[1], color='lightgreen')
axes[1].set_title('Income Distribution (After 1%-99% Percentile Outlier Removal)')
axes[1].set_xlabel('Income')

plt.tight_layout()
plt.show()

## Q2: Percentile Method with Different Thresholds

In [ ]:
thresholds = [(1, 99), (5, 95), (10, 90)]
results = []

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, (low_p, high_p) in enumerate(thresholds):
    low = np.percentile(df['Income'], low_p)
    high = np.percentile(df['Income'], high_p)
    
    df_filtered = df[(df['Income'] >= low) & (df['Income'] <= high)].copy()
    
    results.append({
        'Threshold': f'{low_p}th - {high_p}th',
        'Size': len(df_filtered),
        'Mean': df_filtered['Income'].mean(),
        'Std': df_filtered['Income'].std()
    })
    
    sns.histplot(df_filtered['Income'], kde=True, ax=axes[i], color='coral')
    axes[i].set_title(f'{low_p}th-{high_p}th Percentile Filter')
    axes[i].set_xlabel('Income')

plt.tight_layout()
plt.show()

# Display summary table
summary_df = pd.DataFrame(results)
display(summary_df)

## Q3: Winsorization Technique

In [ ]:
# Use same income dataset
df_winsor = df.copy()

# Apply Winsorization at 5th and 95th percentiles
p5 = np.percentile(df_winsor['Income'], 5)
p95 = np.percentile(df_winsor['Income'], 95)

df_winsor['Income_Winsorized'] = df_winsor['Income'].clip(lower=p5, upper=p95)

print(f"5th Percentile: {p5:,.2f}")
print(f"95th Percentile: {p95:,.2f}")

In [ ]:
# Compare statistics
comparison = pd.DataFrame({
    'Metric': ['Mean', 'Median', 'Std Dev'],
    'Original': [df['Income'].mean(), df['Income'].median(), df['Income'].std()],
    'Winsorized': [df_winsor['Income_Winsorized'].mean(), 
                   df_winsor['Income_Winsorized'].median(), 
                   df_winsor['Income_Winsorized'].std()]
})

display(comparison)

In [ ]:
# Boxplots before and after Winsorization
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

sns.boxplot(y=df['Income'], ax=axes[0], color='skyblue')
axes[0].set_title('Original Income - Boxplot')

sns.boxplot(y=df_winsor['Income_Winsorized'], ax=axes[1], color='lightgreen')
axes[1].set_title('Winsorized Income (5%-95%) - Boxplot')

plt.tight_layout()
plt.show()

## Q4: Winsorization vs Z-score Removal

In [ ]:
# New dataset: Heights with extreme unrealistic outliers
np.random.seed(42)
heights = np.random.normal(loc=170, scale=10, size=1000)

# Add extreme outliers
heights = np.append(heights, [50, 60, 300, 320, 350])

df_height = pd.DataFrame({'Height': heights})
print("Original Height Dataset:")
print(df_height.describe())

In [ ]:
# Method 1: Z-score removal (|Z| > 3)
z_scores = np.abs((df_height['Height'] - df_height['Height'].mean()) / df_height['Height'].std())
df_z_clean = df_height[z_scores <= 3].copy()

# Method 2: Winsorization at 5th and 95th percentiles
p5_h = np.percentile(df_height['Height'], 5)
p95_h = np.percentile(df_height['Height'], 95)
df_winsor_h = df_height.copy()
df_winsor_h['Height_Winsorized'] = df_winsor_h['Height'].clip(lower=p5_h, upper=p95_h)

In [ ]:
# Comparison
comparison_methods = pd.DataFrame({
    'Method': ['Original', 'Z-score Removal (|Z|>3)', 'Winsorization (5%-95%)'],
    'Dataset Size': [len(df_height), len(df_z_clean), len(df_winsor_h)],
    'Mean': [df_height['Height'].mean(), df_z_clean['Height'].mean(), df_winsor_h['Height_Winsorized'].mean()],
    'Median': [df_height['Height'].median(), df_z_clean['Height'].median(), df_winsor_h['Height_Winsorized'].median()]
})

display(comparison_methods)

In [ ]:
# Boxplots comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

sns.boxplot(y=df_height['Height'], ax=axes[0], color='salmon')
axes[0].set_title('Original Heights')

sns.boxplot(y=df_z_clean['Height'], ax=axes[1], color='lightblue')
axes[1].set_title('After Z-score Removal')

sns.boxplot(y=df_winsor_h['Height_Winsorized'], ax=axes[2], color='lightgreen')
axes[2].set_title('After Winsorization (5-95%)')

plt.tight_layout()
plt.show()